# Advanced Reasoning Patterns for AI Agents

**Author:** Shuvam Banerji Seal

**Last Updated:** April 2026

## Learning Objectives

Explore advanced reasoning strategies that enable agents to solve complex problems:

1. Chain-of-Thought reasoning
2. Tree-of-Thought exploration
3. ReACT (Reasoning + Acting)
4. Plan-Execute-Verify patterns
5. Ensemble reasoning

## 1. Chain-of-Thought (CoT) Reasoning

The agent explains its reasoning step-by-step before answering.

### How It Works

```
Problem: "If John has 5 apples and Mary has 3 apples, 
         how many more apples does John have?"

Without CoT:
Answer: "2 apples"

With CoT:
Reasoning:
  1. John has 5 apples
  2. Mary has 3 apples
  3. Difference = 5 - 3 = 2
Answer: "John has 2 more apples"
```

### Python Implementation

```python
def chain_of_thought(agent, problem):
    prompt = f"""
    Solve this problem step-by-step:
    {problem}
    
    Please:
    1. Break down the problem
    2. Show your reasoning
    3. Arrive at the conclusion
    """
    
    response = agent.run(prompt)
    return response
```

### When to Use
- Complex reasoning tasks
- Mathematical problems
- Logic puzzles
- Analysis tasks

## 2. Tree-of-Thought (ToT) Reasoning

Explore multiple reasoning paths and select the best one.

### How It Works

```
                    Problem
                       │
            ┌──────────┼──────────┐
            ▼          ▼          ▼
        Path 1     Path 2     Path 3
          │          │          │
       ┌──┴──┐   ┌──┴──┐   ┌──┴──┐
       ▼     ▼   ▼     ▼   ▼     ▼
      1.1   1.2  2.1   2.2  3.1   3.2
       │     │    │     │    │     │
       └─────┴────┴─────┴────┴─────┘
               │
               ▼
        Evaluate & Select Best
               │
               ▼
            Solution
```

### Python Implementation

```python
async def tree_of_thought(agent, problem, depth=3, width=3):
    paths = []
    
    # Generate multiple reasoning paths
    for i in range(width):
        prompt = f"""
        Solve this problem using approach {i+1}:
        {problem}
        """
        path = await agent.run(prompt)
        paths.append({
            'path': i,
            'reasoning': path,
            'score': evaluate(path)
        })
    
    # Select best path
    best = max(paths, key=lambda x: x['score'])
    return best['reasoning']
```

### Advantages
- More thorough exploration
- Better solution quality
- Handles ambiguous problems

### Trade-offs
- Higher cost (multiple LLM calls)
- Longer latency
- Requires evaluation metric

## 3. ReACT (Reasoning + Acting)

Interleave reasoning with tool use for grounded problem-solving.

### How It Works

```
Thought: "I need to find the current Bitcoin price"
  │
  ▼
Action: Call web_search("Bitcoin price today")
  │
  ▼
Observation: "Bitcoin is trading at $67,000"
  │
  ▼
Thought: "Now I should compare with historical data"
  │
  ▼
Action: Call get_historical_price("Bitcoin", "2025-04-06")
  │
  ▼
Observation: "Bitcoin was $61,000 a year ago"
  │
  ▼
Thought: "So Bitcoin has grown ~10% year-over-year"
Final Answer: "Bitcoin is trading at $67,000, up from $61,000 a year ago"
```

### Implementation

```python
class ReACTAgent:
    def __init__(self, llm, tools):
        self.llm = llm
        self.tools = tools
        self.history = []
    
    async def run(self, query):
        context = f"Question: {query}\n"
        
        for step in range(10):  # max steps
            # Get next action from LLM
            response = self.llm.invoke(context)
            
            # Parse Thought, Action, or Final Answer
            if "Final Answer" in response:
                return response
            
            # Execute action
            action = parse_action(response)
            observation = await self.execute_tool(action)
            
            # Update context
            context += f"{response}\nObservation: {observation}\n"
        
        return "Max steps reached"
```

## 4. Plan-Execute-Verify Pattern

Structure complex tasks into distinct phases.

### How It Works

```
┌──────────────────┐
│ PLAN              │  Agent creates detailed plan
│ - Break into steps│
│ - Identify tools  │
│ - Check feasibility
└────────┬──────────┘
         │
         ▼
┌──────────────────┐
│ EXECUTE          │  Execute plan step by step
│ - Run each step  │
│ - Handle errors  │
│ - Adapt as needed
└────────┬──────────┘
         │
         ▼
┌──────────────────┐
│ VERIFY           │  Check results
│ - Validate output│
│ - Compare to goal
│ - Identify gaps
└────────┬──────────┘
         │
     Goal Met?
    /          \
  YES           NO
   │             │
┌──▼──┐      ┌──▼──────┐
│Done │      │Replan   │──┐
└─────┘      └─────────┘  │
                          │
                         Back to PLAN
```

### Example

**Task**: "Write a research paper on AI agents"

**Plan Phase**:
1. Search for recent papers on AI agents
2. Analyze findings
3. Outline structure
4. Write sections
5. Review and edit

**Execute Phase**:
1. Perform web searches
2. Analyze and take notes
3. Create detailed outline
4. Write each section
5. Compile final paper

**Verify Phase**:
- Check accuracy of facts
- Ensure proper citations
- Review for coherence
- Meet length requirements

## 5. Ensemble Reasoning

Use multiple agents with different perspectives and combine results.

### How It Works

```
                  Problem
                     │
     ┌───────────────┼───────────────┐
     │               │               │
     ▼               ▼               ▼
┌─────────────┐ ┌─────────────┐ ┌─────────────┐
│ Analyst     │ │ Developer   │ │ Critic      │
│ (Objective) │ │ (Practical) │ │ (Skeptical) │
└──────┬──────┘ └──────┬──────┘ └──────┬──────┘
       │               │               │
       ▼               ▼               ▼
   Analysis       Implementation    Critique
       │               │               │
       └───────────────┼───────────────┘
                       │
                       ▼
            Combine & Synthesize
                       │
                       ▼
              Final Decision
```

### Python Implementation

```python
async def ensemble_reasoning(problem):
    # Create agents with different roles
    analyst = Agent(role="Objective Analyst")
    developer = Agent(role="Practical Developer")
    critic = Agent(role="Critical Thinker")
    
    # Get perspectives
    results = await asyncio.gather(
        analyst.run(problem),
        developer.run(problem),
        critic.run(problem)
    )
    
    # Synthesize
    synthesizer = Agent(role="Decision Maker")
    synthesis_prompt = f"""
    Perspectives on: {problem}
    
    Analyst: {results[0]}
    Developer: {results[1]}
    Critic: {results[2]}
    
    Synthesize into best decision.
    """
    
    return await synthesizer.run(synthesis_prompt)
```

## Comparison of Reasoning Patterns

| Pattern | Complexity | Cost | Quality | Latency | Best For |
|---------|-----------|------|---------|---------|----------|
| **Chain-of-Thought** | Low | Low | Medium | Low | Simple reasoning |
| **Tree-of-Thought** | High | High | High | High | Complex problems |
| **ReACT** | Medium | Medium | High | Medium | Tool-based tasks |
| **Plan-Execute-Verify** | High | Medium | Very High | High | Complex workflows |
| **Ensemble** | Very High | Very High | Very High | Very High | Critical decisions |

## Real-World Applications

### 1. Research & Analysis
- **Pattern**: Chain-of-Thought + ReACT
- **Example**: Literature review with tool-assisted searches

### 2. Software Development
- **Pattern**: Plan-Execute-Verify
- **Example**: Code generation with testing loops

### 3. Strategic Decision Making
- **Pattern**: Ensemble + Tree-of-Thought
- **Example**: Business strategy with multiple perspectives

### 4. Customer Support
- **Pattern**: ReACT + Chain-of-Thought
- **Example**: Issue resolution with knowledge base access

### 5. Scientific Discovery
- **Pattern**: Tree-of-Thought + Ensemble
- **Example**: Hypothesis generation and testing

## Cost-Benefit Analysis

### Token Usage Comparison

```
Baseline CoT:         ~500 tokens
Tree-of-Thought:     ~3000 tokens (6x more)
ReACT (5 steps):     ~2000 tokens (4x more)
Plan-Execute-Verify: ~2500 tokens (5x more)
Ensemble (3 agents): ~1500 tokens (3x more)
```

### Quality Improvement

```
Baseline CoT:         70% accuracy
Tree-of-Thought:      92% accuracy (+22%)
ReACT:                88% accuracy (+18%)
Plan-Execute-Verify:  90% accuracy (+20%)
Ensemble:             95% accuracy (+25%)
```

### ROI

```
Low stakes task:    Use CoT (cost optimal)
Medium stakes:      Use ReACT or Plan-Execute
High stakes:        Use Ensemble + Tree-of-Thought
```

## Best Practices

1. **Match Pattern to Problem**
   - Simple problems: Chain-of-Thought
   - Complex problems: Tree-of-Thought or Ensemble
   - Tool-dependent: ReACT
   - Structured workflows: Plan-Execute-Verify

2. **Monitor Cost vs Quality**
   - Track tokens spent vs accuracy gained
   - Optimize for your use case
   - Consider real-time vs batch processing

3. **Combine Patterns**
   - ReACT within plan steps
   - Tree-of-Thought for critical decisions
   - Ensemble for high-stakes

4. **Evaluate Results**
   - Always validate against ground truth
   - Test with diverse inputs
   - Monitor production performance

## Exercise: Implement Hybrid Reasoning

### Task
Build an agent that:
1. Plans approach (Plan phase)
2. Uses tools to gather info (ReACT pattern)
3. Explores alternatives (Tree-of-Thought)
4. Verifies quality (Verify phase)

### Example Problem
"Design a mobile app for a startup. What features should it have? What are the technical considerations?"

### Requirements
- Multi-step reasoning
- Tool integration
- Multiple perspectives
- Validation layer

## Key Takeaways

- **Chain-of-Thought**: Foundation for all reasoning
- **Tree-of-Thought**: For exploration of alternatives
- **ReACT**: Best for grounded, tool-based tasks
- **Plan-Execute-Verify**: Ideal for structured workflows
- **Ensemble**: For critical decisions needing diverse views
- **Combinations**: Most powerful approach
- **Trade-offs**: Always balance quality, cost, and latency
- **Measurement**: Continuously evaluate real-world performance